# Explainability Model Selection

This notebook selects models for the explainability stage. The goal is not only to find the highest single score, but to choose a non-transformer model that is strong and stable across the three target experiments, then compare it against the available transformer-based model.

Target settings:
- `Single-TB`
- `Shift-TA_TC-to-TB_10pct`
- `Shift-TA_TB-to-TC_10pct`

Recommended use:
- Pick one CNN/conv model for main explainability and filtering experiments.
- Use `Segformer-mit_b0-imagenet` as the transformer comparison model.
- Apply shared output-level explainability to both models, and add attention-style analysis only for the transformer model as qualitative evidence.

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

PROJECT_ROOT = Path('/users/7/yu001011/csci5527/CSCI5527-final')
RUNS_DIR = PROJECT_ROOT / 'baseline_models_scripts' / 'runs'
MODELS_DIR = RUNS_DIR / 'models'

TARGET_SETTINGS = [
    'Single-TB',
    'Shift-TA_TC-to-TB_10pct',
    'Shift-TA_TB-to-TC_10pct',
]

METRICS = ['Test_IoU', 'Test_F1', 'Test_Recall', 'Test_Prec']
PROJECT_ROOT, RUNS_DIR, MODELS_DIR

(PosixPath('/users/7/yu001011/csci5527/CSCI5527-final'),
 PosixPath('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs'),
 PosixPath('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/models'))

## 2. Load Available Results and Checkpoints

This cell reads all result CSV files in `runs/` and all checkpoint names in `runs/models/`. Later we keep only rows that have a corresponding `.pth` file, because explainability needs a loadable model.

In [2]:
csv_paths = sorted(RUNS_DIR.glob('*.csv'))
model_paths = sorted(MODELS_DIR.glob('*.pth'))
available_model_names = {p.stem for p in model_paths}

display(Markdown(f'Found **{len(csv_paths)}** CSV files and **{len(model_paths)}** checkpoint files.'))
for p in csv_paths:
    display(Markdown(f'- `{p.name}`'))

Found **11** CSV files and **38** checkpoint files.

- `layer2_arch_deeplabv3plus_resnet34.csv`

- `layer2_arch_fpn_resnet34.csv`

- `layer2_efficientnet-b0.csv`

- `layer2_resnet34.csv`

...

- `layer3_arch_FPN.csv`

- `layer3_arch_Segformer.csv`

- `layer3_arch_UnetPlusPlus.csv`

- `layer3_arch_UnetPlusPlus_resnet34.csv`

- `results_Unet_resnet34.csv`

In [3]:
def load_results(paths):
    frames = []
    for path in paths:
        df = pd.read_csv(path)
        df['source_file'] = path.name
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def parse_experiment(exp_name: str):
    exp_name = str(exp_name)
    setting = None
    model_part = exp_name
    for marker in ['_Single-', '_Shift-', '_Multi-Domain']:
        if marker in exp_name:
            model_part, tail = exp_name.split(marker, 1)
            if marker == '_Single-':
                setting = 'Single-' + tail
            elif marker == '_Shift-':
                setting = 'Shift-' + tail
            else:
                setting = 'Multi-Domain'
            break

    parts = model_part.split('-')
    if len(parts) < 3:
        return pd.Series({'arch': None, 'encoder': None, 'weights': None, 'setting': setting})
    return pd.Series({
        'arch': parts[0],
        'encoder': '-'.join(parts[1:-1]),
        'weights': parts[-1],
        'setting': setting,
    })


results = load_results(csv_paths)
parsed = results['Experiment'].apply(parse_experiment)
results = pd.concat([results, parsed], axis=1)
results['has_checkpoint'] = results['Experiment'].isin(available_model_names)
results['is_transformer'] = results['arch'].str.lower().eq('segformer')

target_results = results[
    results['setting'].isin(TARGET_SETTINGS) & results['has_checkpoint']
].copy()

display(target_results[['Experiment', 'arch', 'encoder', 'weights', 'setting', *METRICS, 'source_file']].sort_values(['setting', 'arch', 'encoder']))

,Experiment,arch,encoder,weights,setting,Test_IoU,Test_F1,Test_Recall,Test_Prec,source_file
0,DeepLabV3Plus-resnet34-imagenet_Shift-TA_TB-to...,DeepLabV3Plus,resnet34,imagenet,Shift-TA_TB-to-TC_10pct,0.357259,0.522876,0.706053,0.426146,layer2_arch_deeplabv3plus_resnet34.csv
12,DeepLabV3Plus-resnet34-imagenet_Shift-TA_TB-to...,DeepLabV3Plus,resnet34,imagenet,Shift-TA_TB-to-TC_10pct,0.374889,0.545063,0.555399,0.535539,layer3_arch_DeepLabV3Plus.csv
1,FPN-resnet34-imagenet_Shift-TA_TB-to-TC_10pct,FPN,resnet34,imagenet,Shift-TA_TB-to-TC_10pct,0.417611,0.586598,0.569568,0.612947,layer2_arch_fpn_resnet34.csv
15,FPN-resnet34-imagenet_Shift-TA_TB-to-TC_10pct,FPN,resnet34,imagenet,Shift-TA_TB-to-TC_10pct,0.313957,0.472811,0.485922,0.461529,layer3_arch_FPN.csv
18,Segformer-mit_b0-imagenet_Shift-TA_TB-to-TC_10pct,Segformer,mit_b0,imagenet,Shift-TA_TB-to-TC_10pct,0.373333,0.542901,0.690581,0.453287,layer3_arch_Segformer.csv
4,Unet-efficientnet-b0-imagenet_Shift-TA_TB-to-T...,Unet,efficientnet-b0,imagenet,Shift-TA_TB-to-TC_10pct,0.377296,0.545845,0.586488,0.510535,layer2_efficientnet-b0.csv
29,Unet-resnet34-imagenet_Shift-TA_TB-to-TC_10pct,Unet,resnet34,imagenet,Shift-TA_TB-to-TC_10pct,0.345064,0.512940,0.580492,0.462844,results_Unet_resnet34.csv
9,Unet-resnet50-imagenet_Shift-TA_TB-to-TC_10pct,Unet,resnet50,imagenet,Shift-TA_TB-to-TC_10pct,0.264646,0.416388,0.433505,0.424765,layer2_resnet50.csv
21,UnetPlusPlus-resnet34-imagenet_Shift-TA_TB-to-...,UnetPlusPlus,resnet34,imagenet,Shift-TA_TB-to-TC_10pct,0.298910,0.453140,0.442815,0.482123,layer3_arch_UnetPlusPlus.csv
24,UnetPlusPlus-resnet34-imagenet_Shift-TA_TB-to-...,UnetPlusPlus,resnet34,imagenet,Shift-TA_TB-to-TC_10pct,0.298909,0.453146,0.442893,0.482008,layer3_arch_UnetPlusPlus_resnet34.csv


## 3. Duplicate Audit

Some experiments appear in more than one CSV. This notebook keeps the best row by `Test_IoU` for each exact experiment name. If duplicate values are nearly identical, this is just cleaning repeated runs. If they differ meaningfully, inspect the `source_file` column before making a final choice.

In [4]:
duplicate_counts = target_results['Experiment'].value_counts()
duplicates = duplicate_counts[duplicate_counts > 1]

if duplicates.empty:
    display(Markdown('No duplicate experiment rows in the target subset.'))
else:
    duplicate_rows = target_results[target_results['Experiment'].isin(duplicates.index)]
    display(duplicate_rows[['Experiment', 'setting', *METRICS, 'source_file']].sort_values(['Experiment', 'Test_IoU'], ascending=[True, False]))

deduped = (
    target_results
    .sort_values(['Experiment', 'Test_IoU', 'Test_F1'], ascending=[True, False, False])
    .drop_duplicates('Experiment', keep='first')
    .copy()
)

display(Markdown(f'Using **{len(deduped)}** unique checkpoint-backed experiment rows after de-duplication.'))

,Experiment,setting,Test_IoU,Test_F1,Test_Recall,Test_Prec,source_file
12,DeepLabV3Plus-resnet34-imagenet_Shift-TA_TB-to...,Shift-TA_TB-to-TC_10pct,0.374889,0.545063,0.555399,0.535539,layer3_arch_DeepLabV3Plus.csv
0,DeepLabV3Plus-resnet34-imagenet_Shift-TA_TB-to...,Shift-TA_TB-to-TC_10pct,0.357259,0.522876,0.706053,0.426146,layer2_arch_deeplabv3plus_resnet34.csv
1,FPN-resnet34-imagenet_Shift-TA_TB-to-TC_10pct,Shift-TA_TB-to-TC_10pct,0.417611,0.586598,0.569568,0.612947,layer2_arch_fpn_resnet34.csv
15,FPN-resnet34-imagenet_Shift-TA_TB-to-TC_10pct,Shift-TA_TB-to-TC_10pct,0.313957,0.472811,0.485922,0.461529,layer3_arch_FPN.csv
21,UnetPlusPlus-resnet34-imagenet_Shift-TA_TB-to-...,Shift-TA_TB-to-TC_10pct,0.298910,0.453140,0.442815,0.482123,layer3_arch_UnetPlusPlus.csv
24,UnetPlusPlus-resnet34-imagenet_Shift-TA_TB-to-...,Shift-TA_TB-to-TC_10pct,0.298909,0.453146,0.442893,0.482008,layer3_arch_UnetPlusPlus_resnet34.csv
20,UnetPlusPlus-resnet34-imagenet_Shift-TA_TC-to-...,Shift-TA_TC-to-TB_10pct,0.308336,0.471008,0.571924,0.419555,layer3_arch_UnetPlusPlus.csv
6,UnetPlusPlus-resnet34-imagenet_Shift-TA_TC-to-...,Shift-TA_TC-to-TB_10pct,0.308278,0.470939,0.571804,0.419581,layer2_resnet34.csv
23,UnetPlusPlus-resnet34-imagenet_Shift-TA_TC-to-...,Shift-TA_TC-to-TB_10pct,0.308235,0.470894,0.571863,0.419408,layer3_arch_UnetPlusPlus_resnet34.csv
22,UnetPlusPlus-resnet34-imagenet_Single-TB,Single-TB,0.319969,0.484504,0.433426,0.555844,layer3_arch_UnetPlusPlus_resnet34.csv


Using **21** unique checkpoint-backed experiment rows after de-duplication.

## 4. Non-Transformer Candidate Ranking

For explainability, the selected non-transformer model should be strong on average and not collapse on a domain-shift setting. The table ranks complete candidates by:

1. mean `Test_IoU` across the three settings
2. worst-setting `Test_IoU`
3. mean `Test_F1`

The `n_settings` column shows whether that architecture/encoder has all three target settings available.

In [5]:
non_transformer = deduped[~deduped['is_transformer']].copy()

candidate_summary = (
    non_transformer
    .groupby(['arch', 'encoder', 'weights'], dropna=False)
    .agg(
        n_settings=('setting', 'nunique'),
        mean_iou=('Test_IoU', 'mean'),
        min_iou=('Test_IoU', 'min'),
        std_iou=('Test_IoU', 'std'),
        mean_f1=('Test_F1', 'mean'),
        mean_recall=('Test_Recall', 'mean'),
        mean_precision=('Test_Prec', 'mean'),
    )
    .reset_index()
)
candidate_summary['complete'] = candidate_summary['n_settings'].eq(len(TARGET_SETTINGS))
candidate_summary['stability_score'] = candidate_summary['mean_iou'] - candidate_summary['std_iou'].fillna(0)

ranked_candidates = candidate_summary.sort_values(
    ['complete', 'mean_iou', 'min_iou', 'mean_f1'],
    ascending=[False, False, False, False],
)

display(ranked_candidates)

,arch,encoder,weights,n_settings,mean_iou,min_iou,std_iou,mean_f1,mean_recall,mean_precision,complete,stability_score
2,Unet,efficientnet-b0,imagenet,3,0.338859,0.269279,0.060369,0.501628,0.551928,0.482349,True,0.278491
3,Unet,resnet34,imagenet,3,0.332405,0.295580,0.032407,0.490276,0.585701,0.438476,True,0.299999
0,DeepLabV3Plus,resnet34,imagenet,3,0.314107,0.259887,0.057781,0.472074,0.543664,0.434174,True,0.256326
5,UnetPlusPlus,resnet34,imagenet,3,0.309072,0.298910,0.010549,0.469551,0.482721,0.485841,True,0.298523
4,Unet,resnet50,imagenet,3,0.302105,0.264646,0.035724,0.458667,0.495247,0.445403,True,0.266381
1,FPN,resnet34,imagenet,3,0.299558,0.236112,0.102332,0.452692,0.510955,0.436322,True,0.197226


In [6]:
pivot_iou = non_transformer.pivot_table(
    index=['arch', 'encoder', 'weights'],
    columns='setting',
    values='Test_IoU',
    aggfunc='max',
)
pivot_f1 = non_transformer.pivot_table(
    index=['arch', 'encoder', 'weights'],
    columns='setting',
    values='Test_F1',
    aggfunc='max',
)

display(Markdown('### Test IoU by setting'))
display(pivot_iou.reindex(columns=TARGET_SETTINGS).sort_values(TARGET_SETTINGS, ascending=False))

display(Markdown('### Test F1 by setting'))
display(pivot_f1.reindex(columns=TARGET_SETTINGS).sort_values(TARGET_SETTINGS, ascending=False))

### Test IoU by setting

setting                                 Single-TB  Shift-TA_TC-to-TB_10pct  \
arch          encoder         weights                                        
Unet          resnet34        imagenet   0.356572                 0.295580   
              resnet50        imagenet   0.335794                 0.305875   
UnetPlusPlus  resnet34        imagenet   0.319969                 0.308336   
DeepLabV3Plus resnet34        imagenet   0.307545                 0.259887   
Unet          efficientnet-b0 imagenet   0.269279                 0.370003   
FPN           resnet34        imagenet   0.244951                 0.236112   

setting                                 Shift-TA_TB-to-TC_10pct  
arch          encoder         weights                            
Unet          resnet34        imagenet                 0.345064  
              resnet50        imagenet                 0.264646  
UnetPlusPlus  resnet34        imagenet                 0.298910  
DeepLabV3Plus resnet34        imagenet                 0.374889  
Unet          efficientnet-b0 imagenet                 0.377296  
FPN           resnet34        imagenet                 0.417611

### Test F1 by setting

setting                                 Single-TB  Shift-TA_TC-to-TB_10pct  \
arch          encoder         weights                                        
Unet          resnet34        imagenet   0.502834                 0.455053   
              resnet50        imagenet   0.492554                 0.467060   
UnetPlusPlus  resnet34        imagenet   0.484504                 0.471008   
DeepLabV3Plus resnet34        imagenet   0.464689                 0.406470   
Unet          efficientnet-b0 imagenet   0.424301                 0.534739   
FPN           resnet34        imagenet   0.392868                 0.378612   

setting                                 Shift-TA_TB-to-TC_10pct  
arch          encoder         weights                            
Unet          resnet34        imagenet                 0.512940  
              resnet50        imagenet                 0.416388  
UnetPlusPlus  resnet34        imagenet                 0.453140  
DeepLabV3Plus resnet34        imagenet                 0.545063  
Unet          efficientnet-b0 imagenet                 0.545845  
FPN           resnet34        imagenet                 0.586598

## 5. Best Model Per Setting

This view is useful because the final choice may depend on whether you value average robustness or performance on a specific hard transfer direction.

In [7]:
best_per_setting = (
    non_transformer
    .sort_values(['setting', 'Test_IoU', 'Test_F1'], ascending=[True, False, False])
    .groupby('setting', as_index=False)
    .head(5)
)

for setting in TARGET_SETTINGS:
    display(Markdown(f'### {setting}'))
    cols = ['Experiment', 'arch', 'encoder', 'weights', *METRICS, 'source_file']
    display(best_per_setting[best_per_setting['setting'].eq(setting)][cols])

### Single-TB

,Experiment,arch,encoder,weights,Test_IoU,Test_F1,Test_Recall,Test_Prec,source_file
26,Unet-resnet34-imagenet_Single-TB,Unet,resnet34,imagenet,0.356572,0.502834,0.559124,0.468969,results_Unet_resnet34.csv
7,Unet-resnet50-imagenet_Single-TB,Unet,resnet50,imagenet,0.335794,0.492554,0.497126,0.496959,layer2_resnet50.csv
22,UnetPlusPlus-resnet34-imagenet_Single-TB,UnetPlusPlus,resnet34,imagenet,0.319969,0.484504,0.433426,0.555844,layer3_arch_UnetPlusPlus_resnet34.csv
10,DeepLabV3Plus-resnet34-imagenet_Single-TB,DeepLabV3Plus,resnet34,imagenet,0.307545,0.464689,0.489089,0.446495,layer3_arch_DeepLabV3Plus.csv
2,Unet-efficientnet-b0-imagenet_Single-TB,Unet,efficientnet-b0,imagenet,0.269279,0.424301,0.390885,0.486734,layer2_efficientnet-b0.csv


### Shift-TA_TC-to-TB_10pct

,Experiment,arch,encoder,weights,Test_IoU,Test_F1,Test_Recall,Test_Prec,source_file
3,Unet-efficientnet-b0-imagenet_Shift-TA_TC-to-T...,Unet,efficientnet-b0,imagenet,0.370003,0.534739,0.678412,0.449778,layer2_efficientnet-b0.csv
20,UnetPlusPlus-resnet34-imagenet_Shift-TA_TC-to-...,UnetPlusPlus,resnet34,imagenet,0.308336,0.471008,0.571924,0.419555,layer3_arch_UnetPlusPlus.csv
8,Unet-resnet50-imagenet_Shift-TA_TC-to-TB_10pct,Unet,resnet50,imagenet,0.305875,0.467060,0.555109,0.414485,layer2_resnet50.csv
31,Unet-resnet34-imagenet_Shift-TA_TC-to-TB_10pct,Unet,resnet34,imagenet,0.295580,0.455053,0.617487,0.383615,results_Unet_resnet34.csv
11,DeepLabV3Plus-resnet34-imagenet_Shift-TA_TC-to...,DeepLabV3Plus,resnet34,imagenet,0.259887,0.406470,0.586503,0.320488,layer3_arch_DeepLabV3Plus.csv


### Shift-TA_TB-to-TC_10pct

,Experiment,arch,encoder,weights,Test_IoU,Test_F1,Test_Recall,Test_Prec,source_file
1,FPN-resnet34-imagenet_Shift-TA_TB-to-TC_10pct,FPN,resnet34,imagenet,0.417611,0.586598,0.569568,0.612947,layer2_arch_fpn_resnet34.csv
4,Unet-efficientnet-b0-imagenet_Shift-TA_TB-to-T...,Unet,efficientnet-b0,imagenet,0.377296,0.545845,0.586488,0.510535,layer2_efficientnet-b0.csv
12,DeepLabV3Plus-resnet34-imagenet_Shift-TA_TB-to...,DeepLabV3Plus,resnet34,imagenet,0.374889,0.545063,0.555399,0.535539,layer3_arch_DeepLabV3Plus.csv
29,Unet-resnet34-imagenet_Shift-TA_TB-to-TC_10pct,Unet,resnet34,imagenet,0.345064,0.512940,0.580492,0.462844,results_Unet_resnet34.csv
21,UnetPlusPlus-resnet34-imagenet_Shift-TA_TB-to-...,UnetPlusPlus,resnet34,imagenet,0.298910,0.453140,0.442815,0.482123,layer3_arch_UnetPlusPlus.csv


## 6. Transformer Reference: SegFormer

There is only one transformer-style candidate here: `Segformer-mit_b0-imagenet`. Treat it as a fixed reference model rather than another architecture search axis. The experiment question becomes:

> Does a transformer-style model show different explanation patterns than the selected CNN/conv model, and do shared confidence/uncertainty filters help both models or mainly one model?

In [8]:
transformer = deduped[deduped['is_transformer']].copy()
display(transformer[['Experiment', 'arch', 'encoder', 'weights', 'setting', *METRICS, 'source_file']].sort_values('setting'))

selected_non_transformer = ranked_candidates[ranked_candidates['complete']].head(1)
if selected_non_transformer.empty:
    selected_non_transformer = ranked_candidates.head(1)

selected_arch = selected_non_transformer.iloc[0]['arch']
selected_encoder = selected_non_transformer.iloc[0]['encoder']
selected_weights = selected_non_transformer.iloc[0]['weights']

selected_cnn_rows = non_transformer[
    non_transformer['arch'].eq(selected_arch)
    & non_transformer['encoder'].eq(selected_encoder)
    & non_transformer['weights'].eq(selected_weights)
].copy()

comparison = pd.concat([
    selected_cnn_rows.assign(model_family='Selected non-transformer'),
    transformer.assign(model_family='Transformer reference'),
], ignore_index=True)

display(Markdown(f'### Selected non-transformer candidate: `{selected_arch}-{selected_encoder}-{selected_weights}`'))
display(comparison[['model_family', 'Experiment', 'setting', *METRICS]].sort_values(['setting', 'model_family']))

,Experiment,arch,encoder,weights,setting,Test_IoU,Test_F1,Test_Recall,Test_Prec,source_file
18,Segformer-mit_b0-imagenet_Shift-TA_TB-to-TC_10pct,Segformer,mit_b0,imagenet,Shift-TA_TB-to-TC_10pct,0.373333,0.542901,0.690581,0.453287,layer3_arch_Segformer.csv
17,Segformer-mit_b0-imagenet_Shift-TA_TC-to-TB_10pct,Segformer,mit_b0,imagenet,Shift-TA_TC-to-TB_10pct,0.176461,0.293250,0.475758,0.215537,layer3_arch_Segformer.csv
16,Segformer-mit_b0-imagenet_Single-TB,Segformer,mit_b0,imagenet,Single-TB,0.231602,0.371456,0.423070,0.352941,layer3_arch_Segformer.csv


### Selected non-transformer candidate: `Unet-efficientnet-b0-imagenet`

,model_family,Experiment,setting,Test_IoU,Test_F1,Test_Recall,Test_Prec
0,Selected non-transformer,Unet-efficientnet-b0-imagenet_Shift-TA_TB-to-T...,Shift-TA_TB-to-TC_10pct,0.377296,0.545845,0.586488,0.510535
3,Transformer reference,Segformer-mit_b0-imagenet_Shift-TA_TB-to-TC_10pct,Shift-TA_TB-to-TC_10pct,0.373333,0.542901,0.690581,0.453287
1,Selected non-transformer,Unet-efficientnet-b0-imagenet_Shift-TA_TC-to-T...,Shift-TA_TC-to-TB_10pct,0.370003,0.534739,0.678412,0.449778
4,Transformer reference,Segformer-mit_b0-imagenet_Shift-TA_TC-to-TB_10pct,Shift-TA_TC-to-TB_10pct,0.176461,0.293250,0.475758,0.215537
2,Selected non-transformer,Unet-efficientnet-b0-imagenet_Single-TB,Single-TB,0.269279,0.424301,0.390885,0.486734
5,Transformer reference,Segformer-mit_b0-imagenet_Single-TB,Single-TB,0.231602,0.371456,0.423070,0.352941


## 7. Suggested Explainability Experiment Design

Use the selected non-transformer model as the main model for filter/enhancement design, because CNN/conv explanations such as saliency and Grad-CAM are easier to connect to local crack texture and false positives.

Use `Segformer-mit_b0-imagenet` as a transformer reference, not as a second model-selection problem. Since there is only one transformer checkpoint per setting, the fair design is to ask whether it behaves differently under the same data settings and the same output-level filters.

Recommended comparison axes:

- **Output-level explanation for both models:** crack probability map, uncertainty map, TP/FP/FN error map, connected-component confidence.
- **Architecture-specific explanation:** Grad-CAM or saliency for the selected CNN/conv model; attention rollout or attention-like token relevance for SegFormer.
- **Filtering/enhancement:** design filters using output-level quantities only, then apply the exact same filter to both models. This avoids giving SegFormer an unfair transformer-only post-processing advantage.
- **Metrics before/after filtering:** IoU, F1, precision, recall. Expect precision to improve if the filter removes false positives; track recall carefully because thin cracks are easy to remove accidentally.

A clean final story would be:

1. Select the strongest complete non-transformer candidate across the three target settings.
2. Compare it with SegFormer on the same three settings.
3. Use explanation maps to identify common FP/FN patterns.
4. Build a confidence/uncertainty/component-based filter.
5. Report whether the filter improves precision/F1 under domain shift without damaging recall too much.

In [9]:
display(Markdown('## Notebook Recommendation'))

top = ranked_candidates.iloc[0]
display(Markdown(
    f"Recommended non-transformer model: "
    f"`{top['arch']}-{top['encoder']}-{top['weights']}` "
    f"with mean IoU={top['mean_iou']:.4f}, min IoU={top['min_iou']:.4f}, "
    f"and mean F1={top['mean_f1']:.4f} across {int(top['n_settings'])} target settings."
))

display(Markdown(
    "Transformer reference: `Segformer-mit_b0-imagenet`. "
    "Use it to compare explanation behavior and filter transferability, not to choose among transformer variants."
))

## Notebook Recommendation

Recommended non-transformer model: `Unet-efficientnet-b0-imagenet` with mean IoU=0.3389, min IoU=0.2693, and mean F1=0.5016 across 3 target settings.

Transformer reference: `Segformer-mit_b0-imagenet`. Use it to compare explanation behavior and filter transferability, not to choose among transformer variants.